<a href="https://colab.research.google.com/github/Ekimxii/fitness_pose_tracker/blob/main/pose-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pose detection: video → landmarks → angles → reps

Runs MediaPipe Pose on a workout video, turns every frame into the same features as the Kaggle data used in `form_analysis.ipynb` (33 landmarks scaled the same way, plus the 12 joint angles from the shared `features.py`), and counts reps with the shared `reps.py`.

Steps: set up → pick a video → find the pose in each frame → convert to the dataset's format → plot angles over time → count reps → save a copy of the video with the skeleton and rep counter drawn on.

Film from the side or at an angle, with the whole body in frame.

In [ ]:
# Setup: MediaPipe, its pose model, and our shared features.py and reps.py
!pip install -q mediapipe   # also installs OpenCV
!wget -q -nc https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task
!wget -q -nc https://raw.githubusercontent.com/Ekimxii/fitness_pose_tracker/main/features.py
!wget -q -nc https://raw.githubusercontent.com/Ekimxii/fitness_pose_tracker/main/reps.py

In [ ]:
# Pick a video: in Colab a file picker opens; elsewhere, set VIDEO to a file path
VIDEO = "Videos for Testing/5 squats.mov"
try:
    from google.colab import files
    VIDEO = next(iter(files.upload()))
except ImportError:
    pass

In [ ]:
# Find the pose in every frame
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from mediapipe.tasks.python import BaseOptions, vision

def extract_pose(video_path, model="pose_landmarker_full.task"):
    """Run MediaPipe on every frame of a video.

    Returns the frame numbers where a person was found, their landmarks as an (n, 33, 3) array
    (x and y as fractions of the image width and height, z on the same scale as x),
    each landmark's visibility (0-1), the frame rate, and the frame size in pixels.
    """
    options = vision.PoseLandmarkerOptions(base_options=BaseOptions(model_asset_path=model),
                                           running_mode=vision.RunningMode.VIDEO)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frames, points, visibility, size = [], [], [], None
    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        i = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            size = (frame.shape[1], frame.shape[0])   # width, height
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            result = landmarker.detect_for_video(image, int(i * 1000 / fps))
            if result.pose_landmarks:   # frames where nobody is found are skipped
                frames.append(i)
                points.append([[lm.x, lm.y, lm.z] for lm in result.pose_landmarks[0]])
                visibility.append([lm.visibility for lm in result.pose_landmarks[0]])
            i += 1
    cap.release()
    assert frames, f"No person found in {video_path} ({i} frames read)"
    print(f"Person found in {len(frames)} of {i} frames; {size[0]}x{size[1]} at {fps:.0f} fps")
    return np.array(frames), np.array(points), np.array(visibility), fps, size

frames, points, visibility, fps, (width, height) = extract_pose(VIDEO)

In [ ]:
# Convert to the Kaggle dataset's format: same columns, same scaling, same 12 angles
from features import ALL_ANGLES, LANDMARK_COLS, LANDMARKS, add_angles, normalize

pixels = points * np.array([width, height, width])   # the dataset was built from pixel coordinates
poses = pd.DataFrame(normalize(pixels).reshape(len(frames), -1), columns=LANDMARK_COLS)
poses.insert(0, "frame", frames)
poses.insert(1, "time_s", frames / fps)
poses = add_angles(poses)

# How sure MediaPipe is that each joint is in view (0-1). Below about 0.5 the position is a guess,
# so don't judge form on those joints in those frames.
vis = pd.DataFrame(visibility, columns=LANDMARKS)
print("joints that are often hidden:", list(vis.columns[(vis < 0.5).mean() > 0.2]))
poses[list(ALL_ANGLES)].rename(columns=ALL_ANGLES).describe().round(0)

In [ ]:
# Angles over time, on the side of the body facing the camera: each rep should show up as one dip or peak
import matplotlib.pyplot as plt
from reps import facing_side_angle, smooth

SHOW = ["knee", "hip bend", "elbow"]   # pick the joints that matter for the exercise
shown = {joint: facing_side_angle(poses, vis, joint) for joint in SHOW}
shown["torso lean"] = smooth(poses["torso_lean"])
gap = poses["time_s"].diff() > 0.5     # nobody detected for a while: break the lines there

fig, ax = plt.subplots(figsize=(10, 4))
for label, angle in shown.items():
    ax.plot(poses["time_s"], angle.mask(gap), label=label, lw=1.8)
ax.set(xlabel="time (s)", ylabel="angle (degrees)", ylim=(0, 185), title="Joint angles over time")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()

## Count reps

A rep is one trip from the rest position to the work position and back (for squats: standing → bottom → standing). Two signals track this, each running from 0 (rest) to 1 (work):

- **motion**: how far the key joint angles (knee and hip bend for squats) have moved from the typical standing pose toward the typical bottom pose, on the side facing the camera. "Typical" is the median in the Kaggle data.
- **model**: the up/down random forest from `form_analysis.ipynb` (chart 8, trained on the Kaggle data's 12 angles): its probability that each frame is in the work position.

A rep counts only when both signals rise above 0.6 and later both fall below 0.4, so each covers the other's mistakes. On the 5-squat test video, the model alone also counted crouching down near the end of the recording (6 reps). The motion alone got 5 there, but any movement that bends the key joints far enough would fool it. Together they count 5.

In [ ]:
# Count reps: the motion (key joint angles) and the up/down model must agree
import kagglehub
from reps import EXERCISES, check_exercise, count_reps, describe_reps, load_training_data, model_signal, motion_signal

EXERCISE = "squats"   # one of: squats, pushups, pullups, situp, jumping_jacks
train = load_training_data(kagglehub.dataset_download("muhannadtuameh/exercise-recognition"))
check_exercise(poses, train, EXERCISE)

motion = motion_signal(poses, vis, train, EXERCISE)   # 0 = rest pose, 1 = work pose, from the angles
model = model_signal(poses, train, EXERCISE)          # the model's probability of the work pose
reps = count_reps(poses["time_s"], motion, model)
print(f"{len(reps)} reps counted (motion alone: {len(count_reps(poses['time_s'], motion))}, "
      f"model alone: {len(count_reps(poses['time_s'], model))})")
describe_reps(reps, poses, vis, motion, EXERCISE)

In [ ]:
# Both signals over time, with each counted rep shaded
from reps import REST_BELOW, WORK_ABOVE

work = "down" if EXERCISES[EXERCISE][1] == "up" else "up"
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(poses["time_s"], motion.mask(gap), lw=1.8, label="motion (key angles)")
ax.plot(poses["time_s"], model.mask(gap), lw=1.8, label=f"model (chance of '{work}')")
for y in (WORK_ABOVE, REST_BELOW):
    ax.axhline(y, color="#9a9893", lw=1, ls="--")
for k, rep in enumerate(reps.itertuples(), start=1):
    ax.axvspan(rep.start_s, rep.end_s, color="#2a78d6", alpha=0.12, lw=0)
    ax.text((rep.start_s + rep.end_s) / 2, 1.12, str(k), ha="center", fontweight="bold")
ax.set(xlabel="time (s)", ylabel="0 = rest, 1 = work", ylim=(-0.1, 1.25),
       title=f"{len(reps)} {EXERCISE} reps: shaded where both signals agree")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()

In [ ]:
# Save a copy of the video with the skeleton, live angles and rep count drawn on
import os

os.makedirs("Video Output", exist_ok=True)
OUT = os.path.join("Video Output", os.path.splitext(os.path.basename(VIDEO))[0] + "_overlay.mp4")
CONNECTIONS = [(c.start, c.end) for c in vision.PoseLandmarksConnections.POSE_LANDMARKS]
row_of = {f: r for r, f in enumerate(frames)}   # frame number -> row in poses
scale = max(0.7, height / 1200)                 # bigger text on bigger (e.g. vertical phone) videos

def put_text(img, text, org, size=1.0):
    s = scale * size
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, s, (0, 0, 0), int(5 * s), cv2.LINE_AA)        # outline
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, s, (255, 255, 255), int(2 * s), cv2.LINE_AA)

cap = cv2.VideoCapture(VIDEO)
writer = cv2.VideoWriter(OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if i in row_of:
        r = row_of[i]
        xy = (points[r, :, :2] * [width, height]).astype(int)
        seen = visibility[r] > 0.5
        for a, b in CONNECTIONS:
            if seen[a] and seen[b]:
                cv2.line(frame, tuple(map(int, xy[a])), tuple(map(int, xy[b])), (255, 255, 255), 2, cv2.LINE_AA)
        for p in xy[seen]:
            cv2.circle(frame, tuple(map(int, p)), 4, (214, 120, 42), -1, cv2.LINE_AA)   # BGR blue
        for k, (label, angle) in enumerate(shown.items()):
            put_text(frame, f"{label}: {angle.iloc[r]:.0f} deg", (10, int((100 + 34 * k) * scale)), 0.8)
    done = int((reps["end_s"] <= i / fps).sum())   # reps finished by this frame
    put_text(frame, f"Reps: {done}", (10, int(60 * scale)), 1.8)
    writer.write(frame)
    i += 1
cap.release()
writer.release()
print("saved", OUT)

try:
    files.download(OUT)   # Colab: save it to your computer
except NameError:
    pass